# FracAtlas one-shot test evaluation (Kaggle)

Phase 3 protocol for any trained run: **step 1** tunes the threshold on that run's validation
predictions, **then the threshold is committed to git**, then **step 2** scores the test set once.
Step 2 refuses to run unless the threshold is committed.

**Settings:** GPU, Internet On. *Add Input*: the FracAtlas dataset **and** the notebook
output that contains the run folder (with `best.pt`). Run the cells one at a time, not Run All.

In [ ]:
RUN = "p5_convnext_tiny_s44"  # run_name of the checkpoint to evaluate

In [ ]:
import os
import subprocess

WORKDIR = "/kaggle/working/repo"
if not os.path.isdir(WORKDIR):
    url = "https://github.com/Akinchan-Jain-7/fracatlas-fracture-detection.git"
    subprocess.run(["git", "clone", "-q", url, WORKDIR], check=True)
os.chdir(WORKDIR)
os.environ["FRACDET_DATA_DIR"] = "/kaggle/input"
!git log --oneline -1
!pip install -q -e .
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
# Step 1 (validation only): choose the threshold. Outputs go to outputs/eval/{RUN}/tune
!python scripts/tune_threshold.py --find-run {RUN}

In [ ]:
# Step 2 (after threshold.json is committed to results/phase3_{RUN}/): score the test set once
!git pull -q --ff-only && git log --oneline -1
!python scripts/evaluate_test.py --find-run {RUN}